In [ ]:
import pandas as pd

# ============================================================
# CẤU HÌNH 
# ============================================================
MAIN_FILE = "D:\\UIT\\KLTN\\data\\final\\vismishds_phase1_final.csv"
SUB_FILE  = "D:\\UIT\\KLTN\\data\\hard_negative\\hard_negative_final.csv"
OUTPUT_FILE = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"
RANDOM_SEED = 42  
# ============================================================

def main():
    # 1. Đọc dữ liệu
    print("📂 Đọc file...")
    df_main = pd.read_csv(MAIN_FILE)
    df_sub  = pd.read_csv(SUB_FILE)

    print(f"   File chính : {len(df_main):,} rows")
    print(f"   File phụ   : {len(df_sub):,} rows")

    # 2. Lọc nhóm ứng viên: label=1 VÀ data_origin=synthetic
    mask_candidates = (df_main["label"] == 1) & (df_main["data_origin"] == "synthetic")
    df_candidates   = df_main[mask_candidates]
    df_rest         = df_main[~mask_candidates]

    n_sub        = len(df_sub)
    n_candidates = len(df_candidates)

    print(f"\n🔍 Nhóm ứng viên (label=1 & data_origin=synthetic): {n_candidates:,} rows")
    print(f"   Số row cần thay thế (= số row file phụ)        : {n_sub:,} rows")

    # Kiểm tra điều kiện
    if n_sub > n_candidates:
        raise ValueError(
            f"❌ File phụ có {n_sub} rows nhưng nhóm ứng viên chỉ có {n_candidates} rows. "
            "Không đủ để thay thế!"
        )

    # 3. Random sample N row từ nhóm ứng viên để xóa
    df_sampled_out = df_candidates.sample(n=n_sub, random_state=RANDOM_SEED)
    df_kept        = df_candidates.drop(df_sampled_out.index)  # phần ứng viên còn lại (không bị thay)

    print(f"\n✂️  Đã random chọn {n_sub:,} rows từ nhóm ứng viên để xóa")
    print(f"   Ứng viên còn lại (giữ nguyên): {len(df_kept):,} rows")

    # 4. Ghép lại: phần không phải ứng viên + ứng viên còn lại + file phụ
    df_output = pd.concat([df_rest, df_kept, df_sub], ignore_index=True)

    # 5. Shuffle toàn bộ dataset
    df_output = df_output.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"\n🔀 Đã shuffle toàn bộ dataset")
    print(f"   Tổng rows output: {len(df_output):,} rows")

    # 6. Xuất file
    df_output.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Đã lưu file output: {OUTPUT_FILE}")

    # Thống kê nhanh
    print("\n📊 Thống kê label trong output:")
    print(df_output["label"].value_counts().to_string())

    if "data_origin" in df_output.columns:
        print("\n📊 Thống kê data_origin trong output:")
        print(df_output["data_origin"].value_counts().to_string())


if __name__ == "__main__":
    main()

📂 Đọc file...
   File chính : 10,562 rows
   File phụ   : 653 rows

🔍 Nhóm ứng viên (label=1 & data_origin=synthetic): 4,996 rows
   Số row cần thay thế (= số row file phụ)        : 653 rows

✂️  Đã random chọn 653 rows từ nhóm ứng viên để xóa
   Ứng viên còn lại (giữ nguyên): 4,343 rows

🔀 Đã shuffle toàn bộ dataset
   Tổng rows output: 10,562 rows

✅ Đã lưu file output: D:\UIT\KLTN\model\base\temp.csv

📊 Thống kê label trong output:
label
0    5320
1    5242

📊 Thống kê data_origin trong output:
data_origin
synthetic                  7342
real                       2567
synthetic_hard_negative     653


In [ ]:
# ============================================================
# CẤU HÌNH 
# ============================================================
MAIN_FILE   = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"   
SUB_FILE_A  = "D:\\UIT\\KLTN\\data\\external\\vilexnorm\\curated\\vilexnorm_clean_p2p_label0.csv"   
SUB_FILE_B  = "D:\\UIT\\KLTN\\data\\external\\vilexnorm\\curated\\vilexnorm_hard_negative_label0.csv"  
OUTPUT_FILE = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"
RANDOM_SEED = 42
# ============================================================

def align_columns(df_sub: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Chỉ giữ các cột khớp với file chính, thêm cột còn thiếu với giá trị NaN.
    Không thêm cột lạ từ file phụ vào.
    """
    df_aligned = pd.DataFrame(columns=columns)
    for col in columns:
        if col in df_sub.columns:
            df_aligned[col] = df_sub[col].values
        else:
            df_aligned[col] = None
    return df_aligned


def replace_batch(df_main: pd.DataFrame, df_sub: pd.DataFrame,
                  sub_name: str, seed_offset: int) -> pd.DataFrame:
    """
    Thay một batch ứng viên (label=0 & data_origin=synthetic) bằng df_sub.
    """
    mask_candidates = (df_main["label"] == 0) & (df_main["data_origin"] == "synthetic")
    df_candidates   = df_main[mask_candidates]
    df_rest         = df_main[~mask_candidates]

    n_sub        = len(df_sub)
    n_candidates = len(df_candidates)

    print(f"\n📂 Xử lý {sub_name}: {n_sub:,} rows")
    print(f"   Ứng viên còn lại (label=0 & data_origin=synthetic): {n_candidates:,} rows")

    if n_sub > n_candidates:
        raise ValueError(
            f"❌ {sub_name} có {n_sub} rows nhưng nhóm ứng viên chỉ còn {n_candidates} rows. "
            "Không đủ để thay thế!"
        )

    # Random sample N row ứng viên để xóa
    df_sampled_out = df_candidates.sample(n=n_sub, random_state=RANDOM_SEED + seed_offset)
    df_kept        = df_candidates.drop(df_sampled_out.index)

    print(f"   ✂️  Đã xóa {n_sub:,} rows ứng viên")
    print(f"   Ứng viên còn lại sau khi xóa: {len(df_kept):,} rows")

    # Align cột file phụ theo cột file chính
    df_sub_aligned = align_columns(df_sub, df_main.columns.tolist())

    # Ghép lại
    df_result = pd.concat([df_rest, df_kept, df_sub_aligned], ignore_index=True)
    return df_result


def main():
    # 1. Đọc dữ liệu
    print("📂 Đọc file...")
    df_main  = pd.read_csv(MAIN_FILE)
    df_sub_a = pd.read_csv(SUB_FILE_A)
    df_sub_b = pd.read_csv(SUB_FILE_B)

    print(f"   File chính  : {len(df_main):,} rows | cột: {df_main.columns.tolist()}")
    print(f"   File phụ A  : {len(df_sub_a):,} rows | cột: {df_sub_a.columns.tolist()}")
    print(f"   File phụ B  : {len(df_sub_b):,} rows | cột: {df_sub_b.columns.tolist()}")

    # 2. Xử lý file phụ A
    df_after_a = replace_batch(df_main, df_sub_a, sub_name="File phụ A", seed_offset=0)

    # 3. Xử lý file phụ B (dùng output sau khi đã xử lý A)
    df_after_b = replace_batch(df_after_a, df_sub_b, sub_name="File phụ B", seed_offset=1)

    # 4. Shuffle toàn bộ
    df_output = df_after_b.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"\n🔀 Đã shuffle toàn bộ dataset")
    print(f"   Tổng rows output: {len(df_output):,} rows")

    # 5. Xuất file (giữ nguyên cột file chính)
    df_output.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Đã lưu file output: {OUTPUT_FILE}")

    # Thống kê nhanh
    print("\n📊 Thống kê label trong output:")
    print(df_output["label"].value_counts().to_string())

    if "data_origin" in df_output.columns:
        print("\n📊 Thống kê data_origin trong output:")
        print(df_output["data_origin"].value_counts().to_string())


if __name__ == "__main__":
    main()

📂 Đọc file...
   File chính  : 10,562 rows | cột: ['sample_id', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id']
   File phụ A  : 500 rows | cột: ['source_dataset', 'source_file', 'source_row_id', 'split', 'original', 'normalized', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'external_source_type', 'text_variant_type', 'candidate_type', 'hard_case_type', 'review_status', 'filter_reason', 'reject_reason', 'flags']
   File phụ B  : 501 rows | cột: ['sample_id', 'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id', 'external_source_type', 'text_variant_type', 'hard_case_type', 'review_status', 'original', 'normalized']

📂 Xử lý File phụ A: 500 rows
   Ứng viên còn lại (label=0 & data_origin=syntheti

In [8]:
import pandas as pd
df = pd.read_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv")
df = df[(df["data_origin"] == 'synthetic') & (df["label"] == 1)]
df.to_csv("D:\\UIT\\KLTN\\model\\base\\merged_dataset_label1_synthetic.csv", index=False)

In [12]:
df = pd.read_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv")
df = df[['content', 'label']]
df.to_csv("D:\\UIT\\KLTN\\model\\base\\vismishs.csv", index=False)

In [9]:
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# CẤU HÌNH 
# ─────────────────────────────────────────────────────────────────────────────
ORIGINAL_FILE  = "D:\\UIT\\KLTN\\model\\base\\merged_dataset.csv"          
REPLACE_FILE   = "D:\\UIT\\KLTN\\cleaned_paraphrased_dataset_094253.csv"
OUTPUT_FILE    = "D:\\UIT\\KLTN\\model\\base\\temp.csv" 

ID_COL         = "sample_id"             
COLUMNS_TO_REPLACE = None                # None = thay tất cả cột (trừ ID_COL)
                                         # hoặc chỉ định: ["content", "label"]
# ─────────────────────────────────────────────────────────────────────────────


def load_file(path: str) -> pd.DataFrame:
    """Load CSV hoặc Excel tự động theo đuôi file."""
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    return pd.read_csv(path)


def replace_by_sample_id(
    original_file: str,
    replace_file: str,
    output_file: str,
    id_col: str,
    columns_to_replace: list | None = None,
) -> pd.DataFrame:

    df_orig    = load_file(original_file)
    df_replace = load_file(replace_file)

    # Xác định cột sẽ được thay thế
    if columns_to_replace is None:
        columns_to_replace = [c for c in df_replace.columns if c != id_col]

    print(f"File gốc       : {original_file}  →  {len(df_orig)} dòng")
    print(f"File thay thế  : {replace_file}   →  {len(df_replace)} dòng")
    print(f"Cột key        : {id_col}")
    print(f"Cột thay thế   : {columns_to_replace}\n")

    # Đặt sample_id làm index để update nhanh
    df_orig    = df_orig.set_index(id_col)
    df_replace = df_replace.set_index(id_col)[columns_to_replace]

    # Log trước khi thay
    matched_ids = df_replace.index.intersection(df_orig.index)
    print(f"Số dòng sẽ được cập nhật: {len(matched_ids)}")

    # Thực hiện thay thế (chỉ các cột chỉ định)
    df_orig.update(df_replace)

    df_result = df_orig.reset_index()

    # Lưu file
    out = Path(output_file)
    if out.suffix.lower() in (".xlsx", ".xls"):
        df_result.to_excel(output_file, index=False)
    else:
        df_result.to_csv(output_file, index=False)

    print(f"\n✅ Đã lưu kết quả vào: {output_file}  ({len(df_result)} dòng)")
    return df_result


# ─────────────────────────────────────────────────────────────────────────────
# Chạy
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df_updated = replace_by_sample_id(
        original_file      = ORIGINAL_FILE,
        replace_file       = REPLACE_FILE,
        output_file        = OUTPUT_FILE,
        id_col             = ID_COL,
        columns_to_replace = COLUMNS_TO_REPLACE,
    )

    # Preview 5 dòng đầu kết quả
    print("\nPreview kết quả:")
    print(df_updated.head())

File gốc       : D:\UIT\KLTN\model\base\merged_dataset.csv  →  10562 dòng
File thay thế  : D:\UIT\KLTN\cleaned_paraphrased_dataset_094253.csv   →  4333 dòng
Cột key        : sample_id
Cột thay thế   : ['content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'obfuscation_level', 'data_origin', 'source_dataset', 'source_file', 'source_row_id']

Số dòng sẽ được cập nhật: 4333


C:\Users\Nam Hai\AppData\Local\Temp\ipykernel_8392\1754866632.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1453 656 145 ... 'phase1_08888' 144 'phase1_08732']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_orig.update(df_replace)



✅ Đã lưu kết quả vào: D:\UIT\KLTN\model\base\temp.csv  (10562 dòng)

Preview kết quả:
       sample_id                                            content  label  \
0   phase1_01451  (TB) SINH NHẬT VINA - NẠP THẺ CÓ QUÀ! VinaPhon...      0   
1   phase1_00657  Quy khach duoc cong them 8 diem Viettel++ boi ...      0   
2  hardneg_00205  Cuc Thue: Nguyen Thi Xuan can bo sung giay to ...      1   
3   phase1_10459  Dịch vụ Quản lý Danh tính Quốc gia: Tài khoản ...      1   
4   phase1_08523  [VPBank] Tài khoản ngưng hoạt động đột ngột. T...      1   

   has_url  has_phone_number      sender_type           category  \
0        1                 1        brandname         Viễn thông   
1        1                 0        brandname               Khác   
2        0                 0        brandname   Dịch vụ công giả   
3        1                 0  personal_number   Dịch vụ công giả   
4        1                 0        brandname  Giả mạo ngân hàng   

                      obfuscation_l

In [11]:
import pandas as pd
df=pd.read_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv")
df=df[(df["data_origin"] == 'synthetic') & (df["label"] == 1)]
df.count()

sample_id            10
content              10
label                10
has_url              10
has_phone_number     10
sender_type          10
category             10
obfuscation_level    10
data_origin          10
source_dataset       10
source_file          10
source_row_id        10
dtype: int64

In [13]:
import pandas as pd
df = pd.read_csv("D:\\UIT\\KLTN\\model\\base\\temp.csv")
df=df.sample(frac=0.1, random_state=42)
df.to_csv("D:\\UIT\\KLTN\\model\\base\\1.csv", index=False)